### Bài 2: Ứng dụng ANN cho bài toán với dữ liệu diabetes.csv với phương forward-modedifferentitation (đạo hàm tiến), reverse-mode differentiation (Đạo hàm ngược) và hàm ngưỡng Sigmoid. Thực hiện và chú thích kết quả.

## **1. Chuẩn bị dữ liệu**

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler


In [3]:
# 1. Doc Du Lieu
df = pd.read_csv('diabetes.csv')

# Giả sử cột cuối cùng là nhãn (Outcome)

X = df.iloc[:, :-1].values
y = df.iloc[:, -1].values.reshape(-1,1) # Reshape để y có kích thước (N, 1)


# 2. Chia tập train/test
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.2, random_state=42)

# 3. Chuẩn hóa dữ liệu (Quan trọng cho hàm Sigmoid)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [4]:
print("Kích thước X_train: ", X_train.shape)
print("Kích thước y_train: ",y_train.shape)

Kích thước X_train:  (606, 8)
Kích thước y_train:  (606, 1)


---

## **2. Xây dựng các hàm cơ bản**
### Sử dụng lại hàm Sigmoid và thêm hàm đạo hàm của nó (Cần cho bước đạo hàm ngược)

In [6]:
# Hàm kích hoạt Sigmoid
def sigmoid(x):
    return 1 /(1+ np.exp(-x))

# Đạo hàm của hàm Signoid: f'(x) = f(x) * (1- f(x))
def sigmoid_derivative(x):
    return x*(1-x)

## 3. Cấu trúc mạng ANN

Chúng ta sẽ thiết kế một mạng đơn giản:

- **Input Layer:** 8 nơ-ron (tương ứng 8 đặc trưng của dữ liệu diabetes).
- **Hidden Layer:** 1 lớp ẩn (ví dụ 4 nơ-ron).
- **Output Layer:** 1 nơ-ron (ra quyết định 0 hoặc 1).


In [8]:
# Khởi tạo số ngẫu nhiên

# Tham số mạng
input_size = 8
hidden_size = 4
output_size = 1
learning_rate = 0.1

# Khởi tạo trọng số (Weights) và Bias
# W!: Trọng số từ Input -> Hidden
# W2: Trọng số từ Hidden -> Output
W1 = np.random.uniform(-1,1,(input_size, hidden_size))
W2 = np.random.uniform(-1, 1, (hidden_size, output_size))

---

## 4. Thực hiện Forward-mode (Lan truyền tiến) & Reverse-mode (Lan truyền ngược)

Đây là vòng lặp huấn luyện chính.

### Giải thích lý thuyết

- **Forward-mode (Lan truyền tiến):**
  - Tính tổng đầu vào lớp ẩn: $Z_1 = X \cdot W_1$
  - Qua hàm kích hoạt: $A_1 = \sigma(Z_1)$
  - Tính tổng đầu vào lớp ra: $Z_2 = A_1 \cdot W_2$
  - Đầu ra cuối cùng: Output $= \sigma(Z_2)$
- **Reverse-mode (Đạo hàm ngược/Backpropagation):**
  - Tính sai số (Error) = $Y_{that} - Output$
  - Tính độ dốc (Gradient) dựa trên quy tắc chuỗi (Chain Rule) để biết cần điều chỉnh $W_1, W_2$ bao nhiêu nhằm giảm sai số.


In [9]:
epochs = 1000 # Số lần học 
losses = []

for epoch in range(epochs):
     # --- PHASE 1: FORWARD PROPAGATION (Lan truyền tiến) ---
    # Tính toán lớp ẩn
    hidden_layer_input = np.dot(X_train, W1)
    hidden_layer_output = sigmoid(hidden_layer_input)

    # Tính toán lớp đầu ra
    output_layer_input = np.dot(hidden_layer_output, W2)
    predicted_output = sigmoid(output_layer_input)

    # --- PHASE 2: REVERSE PROPAGATION (Đạo hàm ngược) ---
    # 1. Tính sai số (Loss)
    error = y_train - predicted_output
    loss = np.mean(np.square(error)) # Mean Squared Error
    losses.append(loss)

    # 2. Tính đạo hàm (Backpropagation)
    # d_output: Đạo hàm tại lớp ra
    # Công thức: d_cost/d_pred * d_pred/d_z
    d_output = error * sigmoid_derivative(predicted_output)

    # d_hidden: Đạo hàm tại lớp ẩn (Lan truyền lỗi về phía sau)
    error_hidden_layer = d_output.dot(W2.T)
    d_hidden = error_hidden_layer * sigmoid_derivative(hidden_layer_output)

    # 3. Cập nhật trọng số (Gradient Descent)
    W2 += hidden_layer_output.T.dot(d_output) * learning_rate
    W1 += X_train.T.dot(d_hidden) * learning_rate

    if epoch % 100 == 0:
        print(f"Epoch {epoch}, Loss: {loss:.4f}")
    # 

Epoch 0, Loss: 0.2398
Epoch 100, Loss: 0.1448
Epoch 200, Loss: 0.1417
Epoch 300, Loss: 0.1390
Epoch 400, Loss: 0.1372
Epoch 500, Loss: 0.1360
Epoch 600, Loss: 0.1353
Epoch 700, Loss: 0.1347
Epoch 800, Loss: 0.1341
Epoch 900, Loss: 0.1336


---

## 5. Đánh giá kết quả

Sau khi huấn luyện, dùng trọng số $W_1, W_2$ đã học được để dự đoán trên tập Test.


In [10]:
# Dự đoán trên tập test
hidden_output_test = sigmoid(np.dot(X_test, W1))
final_output_test = sigmoid(np.dot(hidden_output_test, W2))

# Chuyển xác suất thành nhãn 0 hoặc 1 (ngưỡng 0.5)
predictions = (final_output_test > 0.5).astype(int)

# Tính độ chính xác
accuracy = np.mean(predictions == y_test)
print(f"Độ chính xác trên tập Test: {accuracy * 100:.2f}%")

Độ chính xác trên tập Test: 73.03%


## Chú thích Kết quả:
- **Loss giảm dần:** Nghĩa là phương pháp Reverse-mode differentation đang hoạt động đúng, giúp mạng "học" từ sai số.
- **Kết quả(Accuracy):** Nếu độ chính xác 73.03% với mạng đơn giản này là chấp nhận được
- **Ý nghĩa:**
    - **Forward:** Giúp đưa ra dự đoán
    - **Reverse:** Giúp mô hình thông minh ohnw bằng cách sửa lỗi sai từ các dự đoán trước đó